# Production LightGBM Optimization (48-Feature Optuna Search)

This notebook transitions the Speech Emotion Recognition (SER) pipeline from an experimental baseline to a deployment-ready architecture. 

### Key Architectural Upgrades:
1. **Full Feature Ingestion:** We drop the Random Forest pruning constraint and ingest all 48 acoustic descriptors, allowing the LightGBM tree-splitter to mathematically determine feature importance naturally.
2. **Bayesian Hyperparameter Tuning:** We replace hardcoded guesses (`max_depth=6`, `learning_rate=0.03`) with an automated **Optuna** search to find the true mathematical ceiling for this tabular dataset.
3. **Leakage Prevention:** We implement a strict 90/5/5 data split, ensuring `StandardScaler` is only fitted on the training fold.
4. **Artifact Serialization:** The notebook automatically saves the trained model, encoder, scaler, and parameter JSON to disk for immediate FastAPI/Flask backend deployment.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import log_loss, classification_report

import warnings
warnings.filterwarnings('ignore')

c:\Users\Minh\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Define the Optuna Objective Function
This function builds and destroys mutated LightGBM models. Optuna will hunt for the exact combination of parameters that yields the lowest `multi_logloss` on the 5% validation set.

In [2]:
def objective(trial, X_train, y_train, X_val, y_val, class_weight_dict):
    param = {
        'objective': 'multiclass',
        'num_class': 6,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'class_weight': class_weight_dict,
        'n_estimators': 1000, # Rely on early stopping to halt training
        
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }

    model = lgb.LGBMClassifier(**param)

    model.fit(
        X_train, 
        y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )

    preds = model.predict_proba(X_val)
    return log_loss(y_val, preds)

### 2. Data Ingestion & Strict Preprocessing
We ingest the dataset directly. The 90/5/5 split guarantees that the model has a holdout validation set for Early Stopping, and an entirely blind test set for the final Classification Report.

In [3]:
# Ensure your terminal/IDE is rooted in the project folder so this path resolves correctly
data_path = os.path.join('dataset', 'all_emotions.csv')

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}. Please check folder structure.")

df = pd.read_csv(data_path)

# Dynamic target detection
target_options = ['emotion', 'Emotion', 'label', 'class', 'target']
target_col = next((opt for opt in target_options if opt in df.columns), df.columns[-1])

# Drop target to isolate ALL 48 acoustic features
X = df.drop(columns=[target_col])
y = df[target_col]

# Encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y.astype(str))

# 90/5/5 Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.10, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Scaling (Fit strictly on Train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Balanced Class Weights
unique_classes = np.unique(y_train)
class_weight_dict = dict(zip(unique_classes, compute_class_weight('balanced', classes=unique_classes, y=y_train)))
print("Data successfully loaded, split, and scaled.")


Data successfully loaded, split, and scaled.


### 3. Execute Bayesian Hyperparameter Hunt
Running 50 distinct trials. Optuna will mathematically isolate the peak parameters for our specific 48 features. *(Note: This cell will take time to execute as it trains 50 separate gradient boosting trees).* 

In [4]:
print(f"Initializing Optuna Hunt across all {X.shape[1]} features...")
study = optuna.create_study(direction='minimize')

study.optimize(
    lambda trial: objective(trial, X_train_scaled, y_train, X_val_scaled, y_val, class_weight_dict), 
    n_trials=50 
)

print("\n" + "="*50)
print(f"🏆 48-FEATURE OPTUNA SEARCH COMPLETE 🏆")
print("="*50)
print(f"Best Validation LogLoss: {study.best_value:.4f}")
print("Best Parameters Discovered:")
print(json.dumps(study.best_params, indent=4))


[I 2026-06-08 15:36:17,222] A new study created in memory with name: no-name-2d53b071-4214-4296-a593-2fe333fb4771


Initializing Optuna Hunt across all 48 features...


  File "c:\Users\Minh\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\Minh\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Minh\AppData\Local\Programs\Python\Python314\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Minh\AppData\Local\Programs\Python\Python314\Lib\subprocess.py", line 1038, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                

KeyboardInterrupt: 

### 4. Train Final Model & Serialize Artifacts
We train the ultimate LightGBM model using the `best_params` discovered above, evaluate it on our blind 5% Test Set, and hard-save all artifacts to disk so the live application backend can load them instantly.

In [ ]:
print("\nTraining final evaluation model using Best Parameters...")
final_model = lgb.LGBMClassifier(
    n_estimators=1000,
    class_weight=class_weight_dict,
    random_state=42,
    n_jobs=-1,
    **study.best_params
)

final_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)

print("\n=================== 48-FEATURE OPTIMIZED TEST REPORT ===================")
y_pred = final_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

# ==========================================
# HARD-SAVE ALL ARTIFACTS TO DISK
# ==========================================
print("\nSerializing Optuna optimization results to disk...")

# Save parameters
with open('best_lightgbm_params.json', 'w') as f:
    json.dump(study.best_params, f, indent=4)
    
# Save model, scaler, and encoder binaries
joblib.dump(final_model, 'ser_optuna_lightgbm.joblib')
joblib.dump(scaler, 'ser_optuna_scaler.joblib')
joblib.dump(le, 'ser_optuna_encoder.joblib')

print("SUCCESS: Parameters, Model, Scaler, and Encoder serialized to current directory.")
